**Imports and data loading**

In [98]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score



# Load dataset
data_file = 'Debernardi et al 2020 data.csv'
data = pd.read_csv(data_file)


**Data Preprocessing**

In [ ]:
# Display basic information about the dataset
print("Data Types and Non-Null Counts:")
print(data.info())

print("\nSummary Statistics:")
print(data.describe(include='all'))

print("\nMissing Values Per Column:")
print(data.isnull().sum())

# Visualize the missing values in the dataset
plt.figure(figsize=(12, 6))
sns.heatmap(data.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values in the Dataset')
plt.show()

In [ ]:

# Drop  columns with significant missing values.
columns_to_drop = ['stage', 'sample_id', 'sample_origin', 'benign_sample_diagnosis', 'REG1A', 'plasma_CA19_9']
columns_to_drop = [col for col in columns_to_drop if col in data.columns]
data.drop(columns=columns_to_drop, inplace=True)
data.head()

# Outlier detection using IQR on numeric columns only
numeric_cols = data.select_dtypes(include=[np.number])
Q1 = numeric_cols.quantile(0.25)
Q3 = numeric_cols.quantile(0.75)
IQR = Q3 - Q1
outliers = ((numeric_cols < (Q1 - 1.5 * IQR)) | (numeric_cols > (Q3 + 1.5 * IQR))).any(axis=1)

# Print outliers
print("Outliers identified:")
print(data[outliers])


In [102]:
# Encode categorical variables
label_encoders = {}
categorical_columns = ['sex', 'patient_cohort']
for column in categorical_columns:
    le = LabelEncoder()
    data[column] = le.fit_transform(data[column])
    label_encoders[column] = le

# Convert diagnosis to binary
data['diagnosis_binary'] = data['diagnosis'].apply(lambda x: 1 if x == 1 else 0)
data.drop('diagnosis', axis=1, inplace=True)



In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(data.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


In [ ]:
# Check for class imbalance
print("Class Distribution in Target Variable:")
print(data['diagnosis_binary'].value_counts())

Feature engineering

In [ ]:
# Create interaction terms for features with moderate correlations
data['LYVE1_REG1B'] = data['LYVE1'] * data['REG1B']
data['LYVE1_TFF1'] = data['LYVE1'] * data['TFF1']
data['REG1B_TFF1'] = data['REG1B'] * data['TFF1']

# Update feature matrix
X = data.drop(columns=['diagnosis_binary'])
y = data['diagnosis_binary']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply SMOTE to the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Check the distribution of the target variable after SMOTE
resampled_class_distribution = pd.Series(y_train_resampled).value_counts()
print("Resampled class distribution:\n", resampled_class_distribution)

# Normalize the resampled training data
numerical_features = ['age', 'creatinine', 'LYVE1', 'REG1B', 'TFF1', 'LYVE1_REG1B', 'LYVE1_TFF1', 'REG1B_TFF1']
scaler = StandardScaler()
X_train_resampled[numerical_features] = scaler.fit_transform(X_train_resampled[numerical_features])

# Normalize the test data
X_test[numerical_features] = scaler.transform(X_test[numerical_features])


**Define Cross Validation**

In [106]:
# Define global cross-validation method and parameter grids
cv_method = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid_log_reg = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [100, 200, 500, 1000]  # Increased max_iter
}

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

param_grid_rf = {
    'n_estimators': [50, 100, 200, 500],
    'max_features': ['sqrt', 'log2'],  # Removed 'auto'
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['linear', 'rbf']
}

n_iter_svm = len(param_grid_svm['C']) * len(param_grid_svm['gamma']) * len(param_grid_svm['kernel'])  # Adjusted n_iter for SVM RandomizedSearchCV

**Models**

In [ ]:
# Initialize a Logistic Regression model
log_reg = LogisticRegression(random_state=42, max_iter=1000)

# Initialize GridSearchCV with global variables
grid_search_log_reg = GridSearchCV(estimator=log_reg, param_grid=param_grid_log_reg, cv=cv_method, n_jobs=-1, verbose=2)

# Fit the model
grid_search_log_reg.fit(X_resampled, y_resampled)

# Best parameters and estimator
best_params_log_reg = grid_search_log_reg.best_params_
best_log_reg = grid_search_log_reg.best_estimator_

# Evaluate the best model on the test set
y_test_pred_log_reg = best_log_reg.predict(X_test)
test_accuracy_log_reg = accuracy_score(y_test, y_test_pred_log_reg)
classification_report_test_log_reg = classification_report(y_test, y_test_pred_log_reg)

print("Best Parameters (Logistic Regression): ", best_params_log_reg)
print("Test Accuracy (Logistic Regression): ", test_accuracy_log_reg)
print("Test Classification Report (Logistic Regression):\n", classification_report_test_log_reg)


In [ ]:
# Initialize a KNN model
knn = KNeighborsClassifier()

# Initialize GridSearchCV with global variables
grid_search_knn = GridSearchCV(estimator=knn, param_grid=param_grid_knn, cv=cv_method, n_jobs=-1, verbose=2)

# Fit the model
grid_search_knn.fit(X_resampled, y_resampled)

# Best parameters and estimator
best_params_knn = grid_search_knn.best_params_
best_knn = grid_search_knn.best_estimator_

# Evaluate the best model on the test set
y_test_pred_knn = best_knn.predict(X_test)
test_accuracy_knn = accuracy_score(y_test, y_test_pred_knn)
classification_report_test_knn = classification_report(y_test, y_test_pred_knn)

print("Best Parameters (KNN): ", best_params_knn)
print("Test Accuracy (KNN): ", test_accuracy_knn)
print("Test Classification Report (KNN):\n", classification_report_test_knn)


In [ ]:
# Initialize a Random Forest model
rf = RandomForestClassifier(random_state=42)

# Initialize RandomizedSearchCV with global variables
rf_random = RandomizedSearchCV(estimator=rf, param_distributions=param_grid_rf, n_iter=50, cv=cv_method, verbose=2, random_state=42, n_jobs=-1)

# Fit the model
rf_random.fit(X_resampled, y_resampled)

# Best parameters and estimator
best_params_rf = rf_random.best_params_
best_rf = rf_random.best_estimator_

# Evaluate the best model on the test set
y_test_pred_rf = best_rf.predict(X_test)
test_accuracy_rf = accuracy_score(y_test, y_test_pred_rf)
classification_report_test_rf = classification_report(y_test, y_test_pred_rf)

print("Best Parameters (Random Forest): ", best_params_rf)
print("Test Accuracy (Random Forest): ", test_accuracy_rf)
print("Test Classification Report (Random Forest):\n", classification_report_test_rf)


In [ ]:
# Initialize an SVM model
svm = SVC()

# Initialize RandomizedSearchCV with global variables
svm_random = RandomizedSearchCV(estimator=svm, param_distributions=param_grid_svm, n_iter=50, cv=cv_method, verbose=2, random_state=42, n_jobs=-1)

# Fit the model
svm_random.fit(X_resampled, y_resampled)

# Best parameters and estimator
best_params_svm = svm_random.best_params_
best_svm = svm_random.best_estimator_

# Evaluate the best model on the test set
y_test_pred_svm = best_svm.predict(X_test)
test_accuracy_svm = accuracy_score(y_test, y_test_pred_svm)
classification_report_test_svm = classification_report(y_test, y_test_pred_svm)

print("Best Parameters (SVM): ", best_params_svm)
print("Test Accuracy (SVM): ", test_accuracy_svm)
print("Test Classification Report (SVM):\n", classification_report_test_svm)


**ROC For Comparison**

In [ ]:
# Function to plot ROC curve
def plot_roc_curve(y_test, y_pred_proba, model_name):
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {auc:.2f})')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()

# Plot ROC curves for each model
plt.figure()

# Logistic Regression
y_pred_proba_log_reg = best_log_reg.predict_proba(X_test)[:, 1]
plot_roc_curve(y_test, y_pred_proba_log_reg, 'Logistic Regression')

# KNN
y_pred_proba_knn = best_knn.predict_proba(X_test)[:, 1]
plot_roc_curve(y_test, y_pred_proba_knn, 'KNN')

# Random Forest
y_pred_proba_rf = best_rf.predict_proba(X_test)[:, 1]
plot_roc_curve(y_test, y_pred_proba_rf, 'Random Forest')

# SVM (if using probability estimates)
y_pred_proba_svm = best_svm.decision_function(X_test)
plot_roc_curve(y_test, y_pred_proba_svm, 'SVM')

plt.show()
